In [1]:
!pip install -q mlflow


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.2/49.2 kB 1.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 2.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 54.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 60.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 38.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.4/208.4 kB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.0/77.0 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.2/132.2 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 879.5/879.5 kB 36.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.3/207.3 kB 8.4 MB/s eta 0:00:00


In [2]:
import os
import pandas as pd
import numpy as np
import mlflow
import mlflow.sklearn
from kaggle_secrets import UserSecretsClient

DATA_DIR = "/kaggle/input/competitions/ieee-fraud-detection"

secrets = UserSecretsClient()
os.environ["MLFLOW_TRACKING_USERNAME"] = "gabas22"
os.environ["MLFLOW_TRACKING_PASSWORD"] = secrets.get_secret("DAGSHUB_TOKEN")
mlflow.set_tracking_uri("https://dagshub.com/gabas22/ML-gabas22-Assignement-2.mlflow")


In [3]:
from sklearn.base import BaseEstimator, TransformerMixin
class FraudPrep(BaseEstimator, TransformerMixin):
    def __init__(self, drop_cols):
        self.drop_cols = drop_cols
    def fit(self, X, y=None):
        self.cat_maps_ = {}
        self.medians_ = {}
        Xt = self._prep(X.drop(columns=[c for c in self.drop_cols if c in X.columns]))
        for c in Xt.columns:
            if Xt[c].dtype == "object":
                cats = Xt[c].fillna("missing").astype(str).unique().tolist()
                self.cat_maps_[c] = {v: i for i, v in enumerate(cats)}
            else:
                self.medians_[c] = Xt[c].median()
        return self
    def transform(self, X):
        Xt = self._prep(X.drop(columns=[c for c in self.drop_cols if c in X.columns]))
        for c, m in self.cat_maps_.items():
            if c in Xt.columns:
                Xt[c] = Xt[c].fillna("missing").astype(str).map(m).fillna(-1).astype(int)
        for c, med in self.medians_.items():
            if c in Xt.columns:
                Xt[c] = Xt[c].fillna(med)
        return Xt.values
    def _prep(self, X):
        X = X.copy()
        X["hour"] = (X["TransactionDT"] // 3600) % 24
        X["day"] = (X["TransactionDT"] // (3600*24)) % 7
        return X

In [4]:
model = mlflow.sklearn.load_model("models:/ieee-fraud-best/1")
print("loaded:", type(model))

loaded: <class 'sklearn.pipeline.Pipeline'>


In [5]:
test_t = pd.read_csv(f"{DATA_DIR}/test_transaction.csv")
test_i = pd.read_csv(f"{DATA_DIR}/test_identity.csv")
test = test_t.merge(test_i, how="left", on="TransactionID")
test = test.rename(columns={c: c.replace("-", "_") for c in test.columns})
print("test shape:", test.shape)

test shape: (506691, 433)


In [6]:
preds = model.predict_proba(test)[:, 1]
print("preds shape:", preds.shape)
print("first 5:", preds[:5])

preds shape: (506691,)
first 5: [0.00154163 0.00338061 0.00197564 0.00210887 0.00048275]


In [7]:
sub = pd.DataFrame({"TransactionID": test["TransactionID"], "isFraud": preds})
sub.to_csv("submission.csv", index=False)
print(sub.head())
print("rows:", len(sub))

   TransactionID   isFraud
0        3663549  0.001542
1        3663550  0.003381
2        3663551  0.001976
3        3663552  0.002109
4        3663553  0.000483
rows: 506691
